<a href="https://colab.research.google.com/github/Areej973/BERT-Exercise-Word-Similarity-/blob/main/BERT_Exercise(Word_Similarity).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT Exercise(Word Similarity)

In this notebook we will practice how to use a pre-trained BERT model from Hugging Face to calculate the similarity of the same word used in different sentences.


## Install the Required Library

## Import Libraries

In [3]:
import torch
from transformers import BertTokenizer, BertModel
import numpy as np

## Load Pre-trained BERT Model and Tokenizer

In [4]:
# Load the "bert-base-uncased" model and tokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Tokenize and Embedding Generate

In [5]:
text1 = "She had a brilliant idea for the science project."
text2 = "The night sky was lit up with brilliant stars."

In [6]:
# Tokenize and encode sentences he
encoded_input1 = tokenizer(text1, return_tensors='pt')
encoded_input2 = tokenizer(text2, return_tensors='pt')

In [7]:
encoded_input1

{'input_ids': tensor([[ 101, 2016, 2018, 1037, 8235, 2801, 2005, 1996, 2671, 2622, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [8]:
# Validate the token
for idx, token_id in enumerate(encoded_input1["input_ids"][0]):
    print(idx, token_id, tokenizer.decode([token_id]))

print('='*20)

for idx, token_id in enumerate(encoded_input2["input_ids"][0]):
    print(idx, token_id, tokenizer.decode([token_id]))

0 tensor(101) [CLS]
1 tensor(2016) she
2 tensor(2018) had
3 tensor(1037) a
4 tensor(8235) brilliant
5 tensor(2801) idea
6 tensor(2005) for
7 tensor(1996) the
8 tensor(2671) science
9 tensor(2622) project
10 tensor(1012) .
11 tensor(102) [SEP]
0 tensor(101) [CLS]
1 tensor(1996) the
2 tensor(2305) night
3 tensor(3712) sky
4 tensor(2001) was
5 tensor(5507) lit
6 tensor(2039) up
7 tensor(2007) with
8 tensor(8235) brilliant
9 tensor(3340) stars
10 tensor(1012) .
11 tensor(102) [SEP]


In [9]:
# Generate Embeddings

device = "cuda"
model = model.to(device)
model = model.eval()
with torch.no_grad():
    output1 = model(**encoded_input1.to(device))
    output2 = model(**encoded_input2.to(device))

In [10]:
output1

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.0732, -0.3846, -0.3220,  ..., -0.2920,  0.1595,  0.3173],
         [ 0.5760, -0.4468, -0.0607,  ..., -0.1604,  0.4426,  0.0726],
         [ 0.5785, -0.2223, -0.2149,  ..., -0.4916,  0.2730,  0.3592],
         ...,
         [ 0.5864, -0.3584, -0.1915,  ...,  0.0989,  0.3055, -0.4167],
         [ 0.5684,  0.0105, -0.2919,  ...,  0.1456, -0.3521, -0.4608],
         [ 0.2646,  0.2181,  0.5644,  ...,  0.3828, -0.5457, -0.2097]]],
       device='cuda:0'), pooler_output=tensor([[-0.8165, -0.3374, -0.7278,  0.5879,  0.5783, -0.0699,  0.8459,  0.2791,
         -0.5189, -0.9999, -0.3944,  0.7874,  0.9761,  0.1603,  0.9101, -0.4650,
          0.1544, -0.6038,  0.2290, -0.2519,  0.6490,  0.9997,  0.2521,  0.2837,
          0.3722,  0.9200, -0.6478,  0.9297,  0.9354,  0.7330, -0.4856,  0.2038,
         -0.9855, -0.0250, -0.6253, -0.9883,  0.3023, -0.7213,  0.1087,  0.0908,
         -0.8958,  0.2887,  0.9998, -0.4721,  0.339

In [11]:
target_word = 'brilliant'
# Extract the embeddings for the word "brilliant" from the hidden states of the two sentences.
## Instead of hard-coding the position of the word.                                         ##
## find a way to dynamically determine the index of "brilliant" in the tokenized sentences. ##
# convert the embedding to numpy array
# Your Code Here
# Function to find the index of a target word in tokenized input
def find_word_index(encoded_input, word):
    # Tokenize the target word to handle subword tokenization correctly
    word_tokens = tokenizer.encode(word, add_special_tokens=False)
    # Search for the first token of the word in the input ids
    for i in range(len(encoded_input["input_ids"][0]) - len(word_tokens) + 1):
        if encoded_input["input_ids"][0][i:i+len(word_tokens)].tolist() == word_tokens:
            return i
    return -1

index1 = find_word_index(encoded_input1, target_word)
index2 = find_word_index(encoded_input2, target_word)

In [12]:
index1

4

In [13]:
# Extract embeddings and convert them to numpy arrays
embedding1 = output1.last_hidden_state.detach().cpu().numpy()[0][index1]
embedding2 = output2.last_hidden_state.detach().cpu().numpy()[0][index2]

In [14]:
# Now 'embedding1' and 'embedding2' are the numpy arrays of the embeddings for the word "brilliant"
print("Embedding from sentence 1:", embedding1)
print("Embedding from sentence 2:", embedding2)

Embedding from sentence 1: [ 3.82670830e-03 -3.76763701e-01 -1.99055858e-02  1.23066939e-01
 -9.17454362e-02 -2.61866152e-01 -2.73250669e-01  3.76215428e-01
 -1.14563918e+00 -5.93646884e-01  6.78726614e-01 -8.91198635e-01
 -4.51290160e-02  7.31596947e-01 -4.25073892e-01  3.00156623e-01
  6.72900081e-01 -1.40766844e-01 -1.15817122e-01  3.70697677e-01
 -4.83322054e-01  2.61209846e-01 -1.08888304e+00  3.78462404e-01
  7.15361178e-01 -5.38833499e-01  1.60678715e-01 -1.29203424e-01
  2.02133596e-01 -4.06576842e-02  1.09605417e-01 -2.93831024e-02
  2.77761132e-01  8.79580826e-02 -6.45677090e-01 -4.28418070e-01
 -3.36855024e-01 -2.29922488e-01 -1.01509500e+00 -1.67931349e-03
 -4.17956322e-01 -5.61925411e-01  5.51263154e-01  6.54941574e-02
  5.08426487e-01  9.92223173e-02 -3.08089972e-01 -2.19081998e-01
  1.11750627e+00 -8.18480134e-01 -5.30493915e-01  1.66415021e-01
 -9.86379609e-02  5.51983491e-02 -1.02809034e-01  5.60070932e-01
 -6.95049226e-01 -8.34430754e-01 -2.87744492e-01 -1.43401092e-0

## Calculate the Embedding

In [15]:
def cosine_similarity(a, b):
    a = a / np.sqrt((a**2).sum(-1))
    b = b / np.sqrt((b**2).sum(-1))
    return np.dot(a, b.T)

In [16]:
cosine_similarity(embedding1, embedding2)

np.float32(0.6323112)